# Factor Model: Step 6 - Time-Series Covariance Estimation

## Objective
Estimate the factor and idiosyncratic covariance matrices using time-series data from Step 5.

### Key Components (Paleologo Chapter 6.3-6.4):
1. **Factor Covariance Matrix (Ω_f)**: Covariance of factor returns over time
2. **Idiosyncratic Covariance Matrix (Ω_ε)**: Covariance of residual returns
3. **Shrinkage Methods**: Ledoit-Wolf, DCC, STVU to improve estimates
4. **Autocorrelation Correction**: Newey-West estimator

### Inputs:
- Factor returns from Step 5 (from cross-sectional regression)
- Idiosyncratic returns (residuals) from Step 5

### Outputs:
- `factor_covariance_matrix.parquet` - Ω_f with shrinkage
- `idiosyncratic_covariance_matrix.parquet` - Ω_ε (diagonal)
- `factor_volatilities.csv` - Factor risk statistics
- `covariance_diagnostics.csv` - Quality metrics

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from scipy import linalg
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Load Factor Returns and Residuals from Step 5

In [ ]:
print("Loading factor returns from Step 5...")

# Load factor returns (f_t) from WLS regression
# These should be saved from Step 5
factor_returns = pd.read_parquet('factor_returns_step5.parquet')

print(f"✓ Factor returns loaded")
print(f"  Shape: {factor_returns.shape}")
print(f"  Date range: {factor_returns.index.min()} to {factor_returns.index.max()}")
print(f"  Number of factors: {len(factor_returns.columns)}")

print("\nFactor Returns Summary:")
display(factor_returns.describe())

# Load idiosyncratic returns (residuals ε_t)
print("\nLoading idiosyncratic returns (residuals)...")
idio_returns = pd.read_parquet('residuals_step5.parquet')

print(f"✓ Idiosyncratic returns loaded")
print(f"  Shape: {idio_returns.shape}")
print(f"  Number of assets: {len(idio_returns.columns)}")

## 2. Empirical Factor Covariance Matrix

### Formula (Paleologo 6.3):
$$\hat{\Omega}_f^{\text{emp}} := T^{-1} \sum_{t=1}^{T} \hat{f}_t \hat{f}_t^T$$

Where:
- $\hat{f}_t$ = estimated factor returns at time t
- $T$ = number of time periods

In [ ]:
print("="*80)
print("COMPUTING EMPIRICAL FACTOR COVARIANCE MATRIX")
print("="*80)

# Compute empirical covariance matrix
T = len(factor_returns)
Omega_f_emp = factor_returns.cov()  # This is equivalent to T^-1 * sum(f_t * f_t')

print(f"\n✓ Empirical covariance matrix computed")
print(f"  Dimensions: {Omega_f_emp.shape}")
print(f"  Number of observations (T): {T}")
print(f"  Number of factors (p): {len(factor_returns.columns)}")
print(f"  Ratio p/T: {len(factor_returns.columns)/T:.4f}")

# Check properties
eigenvalues = np.linalg.eigvalsh(Omega_f_emp)
print(f"\n  Eigenvalue range: [{eigenvalues.min():.6f}, {eigenvalues.max():.6f}]")
print(f"  Condition number: {eigenvalues.max() / eigenvalues.min():.2f}")
print(f"  Is positive definite: {np.all(eigenvalues > 0)}")

# Display diagonal (factor volatilities)
factor_vols = np.sqrt(np.diag(Omega_f_emp))
print(f"\nFactor Volatilities (annualized, assuming daily data):")
factor_vol_df = pd.DataFrame({
    'Factor': Omega_f_emp.columns,
    'Volatility': factor_vols,
    'Annualized Vol': factor_vols * np.sqrt(252)
})
display(factor_vol_df.head(10))

## 3. Factor Covariance Matrix Shrinkage

### Issue (Paleologo 6.3.1):
The factor returns $\hat{f}_t$ are estimates from WLS regression. The covariance matrix of estimates is:
$$\text{var}(\hat{f}_t) = \Omega_f + (\mathbf{B}^T \Omega_\epsilon^{-1} \mathbf{B})^{-1}$$

This means $\text{var}(\hat{f}_t)$ is **biased upward**. We need shrinkage.

### Ledoit-Wolf Shrinkage:
$$\hat{\Omega}_f(\rho) = (1-\rho)\hat{\Omega}_f^{\text{emp}} + \rho \frac{\text{trace}(\hat{\Omega}_f)}{m} \mathbf{I}_m$$

Where $\rho \in (0,1)$ is the shrinkage intensity.

In [ ]:
def ledoit_wolf_shrinkage(returns_df):
    """
    Ledoit-Wolf shrinkage for covariance matrix estimation.
    
    Reference: Ledoit and Wolf (2003, 2004)
    Implementation based on Paleologo Eq. 6.6
    """
    X = returns_df.values
    T, p = X.shape
    
    # Demean
    X_demeaned = X - X.mean(axis=0)
    
    # Empirical covariance
    S = np.cov(X_demeaned, rowvar=False, bias=False)
    
    # Shrinkage target: diagonal with average variance
    mu = np.trace(S) / p
    F = mu * np.eye(p)
    
    # Compute optimal shrinkage intensity
    # Simplified formula (see Ledoit-Wolf 2004)
    X2 = X_demeaned ** 2
    
    # Sample covariance of squared returns
    phi_mat = (X_demeaned.T @ X_demeaned) / T - S
    phi = np.sum(phi_mat ** 2)
    
    # Asymptotic variance
    rho_diag = np.sum(X2.T @ X2) / T - np.sum(np.diag(S) ** 2)
    
    # Off-diagonal terms (simplified)
    rho = rho_diag / T
    
    # Compute gamma
    gamma = np.linalg.norm(S - F, 'fro') ** 2
    
    # Shrinkage intensity
    kappa = (phi - rho) / gamma
    shrinkage = max(0, min(1, kappa / T))
    
    # Shrunk covariance matrix
    S_shrunk = (1 - shrinkage) * S + shrinkage * F
    
    return S_shrunk, shrinkage


print("="*80)
print("APPLYING LEDOIT-WOLF SHRINKAGE")
print("="*80)

Omega_f_shrunk, shrinkage_intensity = ledoit_wolf_shrinkage(factor_returns)

# Convert back to DataFrame
Omega_f_shrunk = pd.DataFrame(
    Omega_f_shrunk, 
    index=factor_returns.columns, 
    columns=factor_returns.columns
)

print(f"\n✓ Ledoit-Wolf shrinkage applied")
print(f"  Shrinkage intensity (ρ): {shrinkage_intensity:.4f}")
print(f"  Interpretation: {shrinkage_intensity*100:.2f}% weight on diagonal target")

# Compare eigenvalues
eig_emp = np.linalg.eigvalsh(Omega_f_emp)
eig_shrunk = np.linalg.eigvalsh(Omega_f_shrunk)

print(f"\nEigenvalue comparison:")
print(f"  Empirical:  min={eig_emp.min():.6f}, max={eig_emp.max():.6f}")
print(f"  Shrunk:     min={eig_shrunk.min():.6f}, max={eig_shrunk.max():.6f}")
print(f"  Condition number improved: {eig_emp.max()/eig_emp.min():.2f} → {eig_shrunk.max()/eig_shrunk.min():.2f}")

## 4. Autocorrelation Correction (Newey-West)

### Issue (Paleologo 6.3.4):
Daily factor returns exhibit mild short-term autocorrelation. The Newey-West estimator corrects for this:

$$\hat{\Omega}_f = C_0 + \sum_{l=1}^{l_{\text{max}}} \left(1 - \frac{l}{1 + l_{\text{max}}}\right)(C_l + C_l^T)$$

Where $[C_l]_{i,j} := \text{cov}(f_{i,t}, f_{j,t-l})$ is the lagged covariance matrix.

In [ ]:
def newey_west_covariance(returns_df, max_lags=5):
    """
    Newey-West autocorrelation-consistent covariance estimator.
    
    Reference: Newey and West (1987), Paleologo Section 6.3.4
    """
    X = returns_df.values
    T, p = X.shape
    
    # Demean
    X_demeaned = X - X.mean(axis=0)
    
    # C_0: contemporaneous covariance
    C_0 = (X_demeaned.T @ X_demeaned) / T
    
    # Initialize
    Omega = C_0.copy()
    
    # Add lagged covariances
    for lag in range(1, max_lags + 1):
        # Bartlett kernel weight
        weight = 1 - lag / (1 + max_lags)
        
        # Lagged covariance: E[X_t * X_{t-lag}']
        X_lagged = X_demeaned[:-lag]
        X_current = X_demeaned[lag:]
        C_lag = (X_current.T @ X_lagged) / T
        
        # Add symmetric correction
        Omega += weight * (C_lag + C_lag.T)
    
    return Omega


print("="*80)
print("APPLYING NEWEY-WEST AUTOCORRELATION CORRECTION")
print("="*80)

# Apply Newey-West to shrunk covariance
# First, we need to reconstruct factor returns that match the shrunk covariance
# For practical purposes, apply Newey-West then shrink

max_lags = 5  # Typical for daily data
Omega_f_nw = newey_west_covariance(factor_returns, max_lags=max_lags)

# Convert to DataFrame
Omega_f_nw = pd.DataFrame(
    Omega_f_nw,
    index=factor_returns.columns,
    columns=factor_returns.columns
)

print(f"\n✓ Newey-West correction applied")
print(f"  Maximum lags: {max_lags}")

# Compare with empirical
diff = Omega_f_nw - Omega_f_emp
print(f"\n  Average adjustment: {np.abs(diff.values).mean():.8f}")
print(f"  Max adjustment: {np.abs(diff.values).max():.8f}")
print(f"  Trace change: {np.trace(diff.values):.8f}")

## 5. Combined Estimator: Newey-West + Ledoit-Wolf

Apply both corrections for best estimate.

In [ ]:
print("="*80)
print("FINAL FACTOR COVARIANCE MATRIX (NW + LW Shrinkage)")
print("="*80)

# Apply Ledoit-Wolf shrinkage to Newey-West estimate
# Create temporary DataFrame for shrinkage function
# Note: This is an approximation - ideally we'd apply both simultaneously

# Method: Use Newey-West, then apply Ledoit-Wolf shrinkage
p = len(factor_returns.columns)
mu = np.trace(Omega_f_nw.values) / p
F = mu * np.eye(p)

# Use same shrinkage intensity from before
Omega_f_final = (1 - shrinkage_intensity) * Omega_f_nw.values + shrinkage_intensity * F

# Convert to DataFrame
Omega_f_final = pd.DataFrame(
    Omega_f_final,
    index=factor_returns.columns,
    columns=factor_returns.columns
)

print(f"\n✓ Final factor covariance matrix computed")
print(f"  Corrections applied: Newey-West + Ledoit-Wolf")

# Verify positive definiteness
eig_final = np.linalg.eigvalsh(Omega_f_final)
print(f"\n  Eigenvalue range: [{eig_final.min():.6f}, {eig_final.max():.6f}]")
print(f"  Is positive definite: {np.all(eig_final > 0)}")
print(f"  Condition number: {eig_final.max() / eig_final.min():.2f}")

# Factor volatilities (final)
factor_vols_final = np.sqrt(np.diag(Omega_f_final))
print(f"\nFinal Factor Volatilities:")
factor_vol_final_df = pd.DataFrame({
    'Factor': Omega_f_final.columns,
    'Daily Vol': factor_vols_final,
    'Annual Vol': factor_vols_final * np.sqrt(252),
    'Empirical Vol': factor_vols,
    'Vol Change %': (factor_vols_final / factor_vols - 1) * 100
})
display(factor_vol_final_df.head(15))

## 6. Idiosyncratic Covariance Matrix

### Formula (Paleologo 6.4):
Estimate the diagonal idiosyncratic covariance matrix $\Omega_\epsilon$ using exponential weighting:

$$\hat{\Omega}_\epsilon = \text{diag}(\mathbf{W} \mathbf{E}^T \mathbf{E})$$

Where:
- $\mathbf{E} \in \mathbb{R}^{n \times T}$ = idiosyncratic returns matrix
- $\mathbf{W}$ = exponential weighting matrix with half-life $\tau$
- $[\mathbf{W}]_{t,t} = \kappa \exp(-t/\tau)$

In [ ]:
print("="*80)
print("ESTIMATING IDIOSYNCRATIC COVARIANCE MATRIX")
print("="*80)

# Exponential weighting parameters
halflife = 60  # 60 days (typical for daily equity data)
tau = halflife / np.log(2)

T_idio = len(idio_returns)
n_assets = len(idio_returns.columns)

print(f"\nParameters:")
print(f"  Half-life: {halflife} days")
print(f"  Decay parameter (τ): {tau:.2f}")
print(f"  Number of assets: {n_assets:,}")
print(f"  Number of observations: {T_idio}")

# Create exponential weights
time_indices = np.arange(T_idio)[::-1]  # Reverse: most recent = 0
weights = np.exp(-time_indices / tau)
weights = weights / weights.sum()  # Normalize so sum = 1

print(f"\n  Weight on most recent day: {weights[-1]:.6f}")
print(f"  Weight on oldest day: {weights[0]:.6f}")
print(f"  Effective sample size: {1/np.sum(weights**2):.1f} days")

# Compute weighted variance (diagonal only)
# EWMA variance: σ²_t = Σ w_i * ε²_i
idio_squared = idio_returns.values ** 2
idio_var = (weights.reshape(-1, 1) * idio_squared).sum(axis=0)

# Create diagonal covariance matrix
Omega_epsilon = np.diag(idio_var)

# Convert to DataFrame (for consistency, though it's diagonal)
Omega_epsilon_df = pd.DataFrame(
    idio_var,
    index=idio_returns.columns,
    columns=['Idio_Variance']
)

print(f"\n✓ Idiosyncratic covariance matrix computed")
print(f"  Matrix dimensions: {n_assets} × {n_assets}")
print(f"  Matrix type: Diagonal")

# Summary statistics
idio_vols = np.sqrt(idio_var)
print(f"\nIdiosyncratic Volatility Statistics (daily):")
print(f"  Mean: {idio_vols.mean():.6f}")
print(f"  Median: {np.median(idio_vols):.6f}")
print(f"  Std: {idio_vols.std():.6f}")
print(f"  Min: {idio_vols.min():.6f}")
print(f"  Max: {idio_vols.max():.6f}")

print(f"\nIdiosyncratic Volatility Statistics (annualized):")
print(f"  Mean: {idio_vols.mean() * np.sqrt(252):.4f}")
print(f"  Median: {np.median(idio_vols) * np.sqrt(252):.4f}")

## 7. Idiosyncratic Covariance Shrinkage

### Formula (Paleologo 6.4.5):
Shrink idiosyncratic variances toward identity:

$$\hat{\Omega}_{\epsilon,\text{shrink}}(\rho) = (1-\rho)\hat{\Omega}_\epsilon + \rho \frac{\text{trace}(\hat{\Omega}_\epsilon)}{n} \mathbf{I}_n$$

In [ ]:
print("="*80)
print("IDIOSYNCRATIC COVARIANCE SHRINKAGE")
print("="*80)

# Shrinkage toward average idiosyncratic variance
shrinkage_idio = 0.1  # Conservative shrinkage for idiosyncratic risk

avg_idio_var = idio_var.mean()
idio_var_shrunk = (1 - shrinkage_idio) * idio_var + shrinkage_idio * avg_idio_var

Omega_epsilon_shrunk = pd.DataFrame(
    idio_var_shrunk,
    index=idio_returns.columns,
    columns=['Idio_Variance_Shrunk']
)

print(f"\n✓ Shrinkage applied")
print(f"  Shrinkage intensity: {shrinkage_idio:.2f}")
print(f"  Target: Average idiosyncratic variance = {avg_idio_var:.8f}")

# Compare
idio_vols_shrunk = np.sqrt(idio_var_shrunk)
vol_change = (idio_vols_shrunk - idio_vols) / idio_vols * 100

print(f"\nVolatility changes:")
print(f"  Mean change: {vol_change.mean():.2f}%")
print(f"  Max increase: {vol_change.max():.2f}%")
print(f"  Max decrease: {vol_change.min():.2f}%")

## 8. Save Results

In [ ]:
print("="*80)
print("SAVING COVARIANCE MATRICES")
print("="*80)

# Save factor covariance matrix
print("\nSaving factor covariance matrix...")
Omega_f_final.to_parquet('factor_covariance_matrix.parquet')
print(f"  ✓ Saved to: factor_covariance_matrix.parquet")

# Save idiosyncratic variances (diagonal elements)
print("\nSaving idiosyncratic covariance matrix...")
Omega_epsilon_shrunk.to_parquet('idiosyncratic_covariance_matrix.parquet')
print(f"  ✓ Saved to: idiosyncratic_covariance_matrix.parquet")

# Save factor volatilities summary
print("\nSaving factor volatility statistics...")
factor_vol_final_df.to_csv('factor_volatilities.csv', index=False)
print(f"  ✓ Saved to: factor_volatilities.csv")

# Save diagnostics
print("\nSaving covariance diagnostics...")
diagnostics = pd.DataFrame({
    'Metric': [
        'Number of Factors',
        'Number of Assets',
        'Number of Observations (T)',
        'Ratio p/T',
        'Factor Cov Shrinkage Intensity',
        'Idio Cov Shrinkage Intensity',
        'Newey-West Lags',
        'EWMA Half-life (days)',
        'Factor Cov Condition Number',
        'Factor Cov Is PD',
        'Avg Factor Vol (annual)',
        'Avg Idio Vol (annual)'
    ],
    'Value': [
        len(factor_returns.columns),
        n_assets,
        T,
        len(factor_returns.columns)/T,
        shrinkage_intensity,
        shrinkage_idio,
        max_lags,
        halflife,
        eig_final.max() / eig_final.min(),
        np.all(eig_final > 0),
        factor_vols_final.mean() * np.sqrt(252),
        idio_vols_shrunk.mean() * np.sqrt(252)
    ]
})
diagnostics.to_csv('covariance_diagnostics.csv', index=False)
print(f"  ✓ Saved to: covariance_diagnostics.csv")

print("\n" + "="*80)
print("✓ All covariance matrices saved successfully")
print("="*80)

## 9. Visualization

In [ ]:
print("Creating visualizations...")

# Plot 1: Factor correlation matrix heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation matrix
factor_corr = Omega_f_final / np.outer(factor_vols_final, factor_vols_final)
sns.heatmap(factor_corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
            square=True, cbar_kws={'label': 'Correlation'}, ax=axes[0])
axes[0].set_title('Factor Correlation Matrix (Shrunk)', fontsize=12, fontweight='bold')

# Factor volatilities
factor_vol_final_df_sorted = factor_vol_final_df.sort_values('Annual Vol', ascending=False).head(20)
axes[1].barh(range(len(factor_vol_final_df_sorted)), factor_vol_final_df_sorted['Annual Vol'])
axes[1].set_yticks(range(len(factor_vol_final_df_sorted)))
axes[1].set_yticklabels(factor_vol_final_df_sorted['Factor'])
axes[1].set_xlabel('Annualized Volatility', fontsize=10)
axes[1].set_title('Top 20 Factor Volatilities', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('factor_covariance_visualization.png', dpi=150, bbox_inches='tight')
print("  ✓ Saved: factor_covariance_visualization.png")
plt.show()

# Plot 2: Idiosyncratic volatility distribution
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.hist(idio_vols_shrunk * np.sqrt(252), bins=50, alpha=0.7, edgecolor='black')
ax.axvline(idio_vols_shrunk.mean() * np.sqrt(252), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {idio_vols_shrunk.mean() * np.sqrt(252):.3f}')
ax.set_xlabel('Annualized Idiosyncratic Volatility', fontsize=11)
ax.set_ylabel('Number of Assets', fontsize=11)
ax.set_title('Distribution of Idiosyncratic Volatilities', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('idiosyncratic_volatility_distribution.png', dpi=150, bbox_inches='tight')
print("  ✓ Saved: idiosyncratic_volatility_distribution.png")
plt.show()

## 10. Summary and Diagnostics

In [ ]:
print("="*80)
print("STEP 6 COMPLETE - COVARIANCE ESTIMATION")
print("="*80)

print(f"\n📊 SUMMARY")
print(f"\nFactor Covariance Matrix:")
print(f"  Dimensions: {Omega_f_final.shape[0]} × {Omega_f_final.shape[1]}")
print(f"  Estimation method: Newey-West + Ledoit-Wolf Shrinkage")
print(f"  Shrinkage intensity: {shrinkage_intensity:.4f}")
print(f"  Autocorrelation lags: {max_lags}")
print(f"  Condition number: {eig_final.max() / eig_final.min():.2f}")
print(f"  Positive definite: {np.all(eig_final > 0)}")

print(f"\nIdiosyncratic Covariance Matrix:")
print(f"  Dimensions: {n_assets} × {n_assets} (diagonal)")
print(f"  Estimation method: EWMA + Shrinkage")
print(f"  EWMA half-life: {halflife} days")
print(f"  Shrinkage intensity: {shrinkage_idio:.2f}")

print(f"\nVolatility Summary (Annualized):")
print(f"  Average factor volatility: {factor_vols_final.mean() * np.sqrt(252):.4f}")
print(f"  Average idiosyncratic volatility: {idio_vols_shrunk.mean() * np.sqrt(252):.4f}")
print(f"  Ratio (idio/factor): {(idio_vols_shrunk.mean() / factor_vols_final.mean()):.2f}")

print(f"\n📁 Output Files:")
print(f"  ✓ factor_covariance_matrix.parquet")
print(f"  ✓ idiosyncratic_covariance_matrix.parquet")
print(f"  ✓ factor_volatilities.csv")
print(f"  ✓ covariance_diagnostics.csv")
print(f"  ✓ factor_covariance_visualization.png")
print(f"  ✓ idiosyncratic_volatility_distribution.png")

print(f"\n" + "="*80)
print("Next: Use covariance matrices for portfolio optimization")
print("="*80)